# Pipeline

In [1]:
import pandas as pd
import sqlite3
import os
import requests
import time
import kagglehub
from bs4 import BeautifulSoup
from tqdm import tqdm

# ==========================================
# 1. FUNCIONES DE EXTRACCIÓN (E)
# ==========================================

def extraer_olist():
    print("📦 Descargando datos de Olist...")
    archivos = ["olist_order_items_dataset.csv", "olist_products_dataset.csv"]
    ruta_base = kagglehub.dataset_download("olistbr/brazilian-ecommerce", force_download=True)
    datasets = {}
    for archivo in archivos:
        ruta_full = os.path.join(ruta_base, archivo)
        nombre = archivo.replace('olist_', '').replace('_dataset', '').replace('.csv', '')
        datasets[nombre] = pd.read_csv(ruta_full, encoding='latin1')
    return datasets

def extraer_scraping(paginas=5):
    print(f"🌐 Web Scraping: Extrayendo {paginas} páginas de competencia...")
    datos_libros = []
    for pagina in tqdm(range(1, paginas + 1), unit="pág"):
        url = f"https://books.toscrape.com/catalogue/page-{pagina}.html"
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                sopa = BeautifulSoup(r.text, 'html.parser')
                for libro in sopa.find_all('article', class_='product_pod'):
                    datos_libros.append({
                        'Titulo': libro.find('h3').find('a')['title'],
                        'Precio_Crudo': libro.find('p', class_='price_color').text
                    })
            time.sleep(0.2)
        except: continue
    return pd.DataFrame(datos_libros)

def extraer_tasas():
    print("💰 Consultando API de tasas de cambio...")
    url = "https://api.frankfurter.app/latest?from=GBP&to=BRL,USD"
    r = requests.get(url)
    datos = r.json()
    return {
        'GBP_USD': datos['rates']['USD'],
        'BRL_USD': datos['rates']['USD'] / datos['rates']['BRL']
    }

# ==========================================
# 2. FUNCIONES DE TRANSFORMACIÓN (T)
# ==========================================

def enmascarar_id(seller_id):
    id_str = str(seller_id)
    return "****-****-****-" + id_str[-4:] if len(id_str) > 4 else "****"

def transformar_todo(df_olist_raw, df_scraping_raw, tasas):
    print("⚙️ Ejecutando transformaciones y reglas de negocio...")
    
    # --- A. Transformar Scraping (Competencia) ---
    df_scrap = df_scraping_raw.copy()
    # Limpieza de duplicados por título antes de asignar IDs
    df_scrap = df_scrap.drop_duplicates(subset=['Titulo'], keep='first')
    df_scrap['Precio_USD'] = df_scrap['Precio_Crudo'].str.extract(r'(\d+\.\d+)').astype(float) * tasas['GBP_USD']
    df_scrap['Precio_USD'] = df_scrap['Precio_USD'].round(2)
    df_scrap['Origen_Datos'] = 'Web_Scraping_Competencia'
    df_scrap['Categoria'] = 'libros_competencia'
    # Generación de PK Sustituta
    df_scrap['ID_Producto'] = ['SCRP-' + str(i).zfill(4) for i in range(1, len(df_scrap) + 1)]
    
    # --- B. Transformar Olist (Interno) ---
    categorias_libros = ['livros_interesse_geral', 'livros_tecnicos', 'livros_importados']
    df_p = df_olist_raw['products']
    df_i = df_olist_raw['order_items']
    
    df_prod_filt = df_p[df_p['product_category_name'].isin(categorias_libros)].copy()
    df_ventas_merged = pd.merge(df_prod_filt, df_i, on='product_id', how='inner')
    
    df_ventas_merged['Precio_USD'] = (df_ventas_merged['price'] * tasas['BRL_USD']).round(2)
    df_ventas_merged['Origen_Datos'] = 'Olist_Interno'
    
    # --- C. Preparación de DataFrames para Carga SQL ---
    
    # 1. Tabla: olist_productos
    productos_sql = df_prod_filt[['product_id', 'product_category_name']]
    
    # 2. Tabla: olist_ventas (con Enmascaramiento)
    ventas_sql = df_ventas_merged[['order_id', 'order_item_id', 'product_id', 'seller_id', 'price']].copy()
    ventas_sql['seller_id'] = ventas_sql['seller_id'].apply(enmascarar_id)
    
    # 3. Tabla: competencia_libros
    scraping_sql = df_scrap[['ID_Producto', 'Categoria', 'Precio_USD', 'Origen_Datos']].copy()
    
    # 4. Tabla: catalogo_maestro (Deduplicado por Precio Máximo)
    olist_para_maestro = df_ventas_merged[['product_id', 'product_category_name', 'Precio_USD', 'Origen_Datos']].copy()
    olist_para_maestro.columns = ['ID_Producto', 'Categoria', 'Precio_USD', 'Origen_Datos']
    
    maestro_sql = pd.concat([olist_para_maestro, scraping_sql], ignore_index=True)
    maestro_sql = maestro_sql.sort_values(by='Precio_USD', ascending=False).drop_duplicates(subset=['ID_Producto'], keep='first')
    
    return productos_sql, ventas_sql, scraping_sql, maestro_sql

# ==========================================
# 3. FUNCIÓN DE CARGA (L)
# ==========================================

def ejecutar_carga(df_p, df_v, df_s, df_m):
    print("📥 Iniciando carga en 'Test2.db'...")
    conn = sqlite3.connect('Test2.db')
    cursor = conn.cursor()
    # Activamos FK para asegurar integridad durante la carga
    cursor.execute("PRAGMA foreign_keys = ON;")
    
    try:
        # ORDEN IMPORTANTE: Productos antes que Ventas por la FK
        
        # 1. Productos Olist
        df_p.to_sql('stg_p', conn, if_exists='replace', index=False)
        cursor.execute("REPLACE INTO olist_productos SELECT * FROM stg_p")
        
        # 2. Ventas Olist (FK validada)
        df_v.to_sql('stg_v', conn, if_exists='replace', index=False)
        cursor.execute("REPLACE INTO olist_ventas SELECT * FROM stg_v")
        
        # 3. Competencia
        df_s.to_sql('stg_s', conn, if_exists='replace', index=False)
        cursor.execute("REPLACE INTO competencia_libros SELECT * FROM stg_s")
        
        # 4. Catálogo Maestro
        df_m.to_sql('stg_m', conn, if_exists='replace', index=False)
        cursor.execute("REPLACE INTO catalogo_maestro SELECT * FROM stg_m")
        
        conn.commit()
        print("✅ Pipeline ETL finalizado con éxito.")

    except Exception as e:
        print(f"❌ Error crítico en la carga: {e}")
    finally:
        # Limpieza de tablas temporales
        for tabla in ['stg_p', 'stg_v', 'stg_s', 'stg_m']:
            cursor.execute(f"DROP TABLE IF EXISTS {tabla}")
        conn.close()

# ==========================================
# BLOQUE PRINCIPAL
# ==========================================

# 1. Extraer
datasets_olist = extraer_olist()
df_scraping_raw = extraer_scraping(paginas=5) # Puedes subirlo a 50
tasas = extraer_tasas()

# 2. Transformar
df_p_sql, df_v_sql, df_s_sql, df_m_sql = transformar_todo(datasets_olist, df_scraping_raw, tasas)

# 3. Cargar
ejecutar_carga(df_p_sql, df_v_sql, df_s_sql, df_m_sql)

C:\Users\JuanSebastiánArbelae\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📦 Descargando datos de Olist...


100%|█████████████████████████████████████████████████████████████████████████████| 42.6M/42.6M [00:12<00:00, 3.49MB/s]

Extracting files...


🌐 Web Scraping: Extrayendo 5 páginas de competencia...


100%|███████████████████████████████████████████████████████████████████████████████████| 5/5 [00:04<00:00,  1.08pág/s]


💰 Consultando API de tasas de cambio...
⚙️ Ejecutando transformaciones y reglas de negocio...
📥 Iniciando carga en 'Test2.db'...
✅ Pipeline ETL finalizado con éxito.


In [3]:
conn = sqlite3.connect('Test2.db')
    
query = f"""SELECT * FROM catalogo_maestro;"""

df_resultado = pd.read_sql(query, conn)
conn.close()

df_resultado.head()

,id_producto,categoria,precio_usd,origen_datos
0,b0c7a71c8620bd389e240f63a507dc50,livros_interesse_geral,180.66,Olist_Interno
1,078b7149a32b479d3cbf1649fea0172c,livros_importados,150.36,Olist_Interno
2,310b40ec41fbfc061e5309006482e68a,livros_interesse_geral,110.39,Olist_Interno
3,1cc61b32763a4d816212b3507b6b6c59,livros_interesse_geral,105.37,Olist_Interno
4,SCRP-0069,libros_competencia,78.50,Web_Scraping_Competencia


In [6]:
import pandas as pd
import sqlite3
import os
import requests
import time
import kagglehub
from bs4 import BeautifulSoup
from tqdm import tqdm

class OlistDataPipeline:
    def __init__(self, db_name='Test2.db', pages_to_scrape=5):
        """
        Inicializa el pipeline de datos con parámetros configurables.
        """
        self.db_name = db_name
        self.pages_to_scrape = pages_to_scrape
        self.raw_data = {}      
        self.processed_data = {} 
        self.tasas = {}         
        
        # Inicializamos la base de datos
        self._setup_db()

    def _setup_db(self):
        """Asegura la existencia del archivo de base de datos."""
        conn = sqlite3.connect(self.db_name)
        conn.close()

    def run_pipeline(self):
        """
        Orquestador principal (Equivalente a train_and_evaluate).
        """
        start_time = time.time()
        print(f"🚀 Iniciando Pipeline de Datos en {self.db_name}")
        
        try:
            self.extract()
            self.transform()
            self.load()
            
            total_time = time.time() - start_time
            print(f"\n✨ PIPELINE FINALIZADO CON ÉXITO EN {total_time:.2f} SEGUNDOS")
        except Exception as e:
            print(f"\n❌ Fallo crítico en el pipeline: {e}")

    def extract(self):
        """Fase de Extracción: Kaggle, Web Scraping y API."""
        print("\n--- [1/3] Extrayendo fuentes de datos ---")
        
        # 1. Olist (Kaggle) con mapeo explícito para evitar KeyError
        archivos_map = {
            "olist_order_items_dataset.csv": "order_items",
            "olist_products_dataset.csv": "products"
        }
        ruta_base = kagglehub.dataset_download("olistbr/brazilian-ecommerce", force_download=True)
        self.raw_data['olist'] = {
            llave: pd.read_csv(os.path.join(ruta_base, csv), encoding='latin1')
            for csv, llave in archivos_map.items()
        }

        # 2. Scraping de libros
        datos_scrap = []
        for p in tqdm(range(1, self.pages_to_scrape + 1), desc="Scraping"):
            r = requests.get(f"https://books.toscrape.com/catalogue/page-{p}.html")
            if r.status_code == 200:
                sopa = BeautifulSoup(r.text, 'html.parser')
                for libro in sopa.find_all('article', class_='product_pod'):
                    datos_scrap.append({
                        'Titulo': libro.find('h3').find('a')['title'],
                        'Precio_Crudo': libro.find('p', class_='price_color').text
                    })
            time.sleep(0.1)
        self.raw_data['scraping'] = pd.DataFrame(datos_scrap)

        # 3. API Tasas (GBP -> BRL/USD)
        res = requests.get("https://api.frankfurter.app/latest?from=GBP&to=BRL,USD").json()
        self.tasas = {
            'GBP_USD': res['rates']['USD'], 
            'BRL_USD': res['rates']['USD'] / res['rates']['BRL']
        }
        print(f"✅ Datos extraídos correctamente.")

    def transform(self):
        """Fase de Transformación: Limpieza y Reglas de Negocio."""
        print("\n--- [2/3] Transformando datos y aplicando lógica ---")
        
        # A. Transformar Scraping (Competencia)
        df_s = self.raw_data['scraping'].copy()
        df_s = df_s.drop_duplicates(subset=['Titulo'])
        df_s['Precio_USD'] = df_s['Precio_Crudo'].str.extract(r'(\d+\.\d+)').astype(float) * self.tasas['GBP_USD']
        df_s['ID_Producto'] = ['SCRP-' + str(i).zfill(4) for i in range(1, len(df_s) + 1)]
        df_s['Origen_Datos'] = 'Web_Scraping_Competencia'
        df_s['Categoria'] = 'libros_competencia'
        
        self.processed_data['competencia'] = df_s[['ID_Producto', 'Categoria', 'Precio_USD', 'Origen_Datos']]

        # B. Transformar Olist (Interno) - CORRECCIÓN DE NOMBRES AQUÍ
        df_p = self.raw_data['olist']['products']
        df_i = self.raw_data['olist']['order_items']
        
        categorias_libros = ['livros_interesse_geral', 'livros_tecnicos', 'livros_importados']
        df_filt = df_p[df_p['product_category_name'].isin(categorias_libros)].copy()
        
        # Uso de df_i (definido arriba) para el merge
        df_m = pd.merge(df_filt, df_i, on='product_id', how='inner')
        df_m['Precio_USD'] = (df_m['price'] * self.tasas['BRL_USD']).round(2)
        
        # Almacenamos tablas individuales para olist_productos y olist_ventas
        self.processed_data['olist_prod'] = df_filt[['product_id', 'product_category_name']]
        
        ventas = df_m[['order_id', 'order_item_id', 'product_id', 'seller_id', 'price']].copy()
        ventas['seller_id'] = ventas['seller_id'].apply(lambda x: "****-****-" + str(x)[-4:])
        self.processed_data['olist_ventas'] = ventas

        # C. Crear Catálogo Maestro (Deduplicado por Precio Máximo)
        olist_m = df_m[['product_id', 'product_category_name', 'Precio_USD']].copy()
        olist_m.columns = ['ID_Producto', 'Categoria', 'Precio_USD']
        olist_m['Origen_Datos'] = 'Olist_Interno'
        
        maestro = pd.concat([olist_m, self.processed_data['competencia']], ignore_index=True)
        self.processed_data['maestro'] = maestro.sort_values('Precio_USD', ascending=False).drop_duplicates('ID_Producto')
        
        print(f"✅ Transformación exitosa.")

    def load(self):
        """Fase de Carga: Inserción Incremental en SQLite."""
        print("\n--- [3/3] Cargando en base de datos ---")
        conn = sqlite3.connect(self.db_name)
        cursor = conn.cursor()
        cursor.execute("PRAGMA foreign_keys = ON;")

        # Mapeo de processed_data a nombres de tablas reales en SQL
        mapeo = {
            'olist_prod': 'olist_productos', 
            'olist_ventas': 'olist_ventas',
            'competencia': 'competencia_libros', 
            'maestro': 'catalogo_maestro'
        }

        try:
            for key, table in mapeo.items():
                df = self.processed_data[key]
                stg = f"stg_{key}"
                
                # Carga incremental vía Staging
                df.to_sql(stg, conn, if_exists='replace', index=False)
                cols = ", ".join(df.columns)
                cursor.execute(f"REPLACE INTO {table} ({cols}) SELECT {cols} FROM {stg}")
                cursor.execute(f"DROP TABLE {stg}")
                print(f"✅ Tabla '{table}' sincronizada.")
            
            conn.commit()
        except Exception as e:
            conn.rollback()
            raise e
        finally:
            conn.close()

# --- EJECUCIÓN ---
if __name__ == "__main__":
    # Creamos el objeto pipeline
    pipeline = OlistDataPipeline(db_name='Test2.db', pages_to_scrape=5)
    
    # Ejecutamos el orquestador
    pipeline.run_pipeline()

🚀 Iniciando Pipeline de Datos en Test2.db

--- [1/3] Extrayendo fuentes de datos ---


100%|█████████████████████████████████████████████████████████████████████████████| 42.6M/42.6M [00:07<00:00, 5.71MB/s]

Extracting files...



Scraping: 100%|██████████████████████████████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.30it/s]


✅ Datos extraídos correctamente.

--- [2/3] Transformando datos y aplicando lógica ---
✅ Transformación exitosa.

--- [3/3] Cargando en base de datos ---
✅ Tabla 'olist_productos' sincronizada.
✅ Tabla 'olist_ventas' sincronizada.
✅ Tabla 'competencia_libros' sincronizada.
✅ Tabla 'catalogo_maestro' sincronizada.

✨ PIPELINE FINALIZADO CON ÉXITO EN 15.51 SEGUNDOS
